# NB09 — Rust: Sequences, K-mers, and FASTA Parsing

**Module 17: HPC, Parallel Computing, and Rust**

---

## 1. Motivation

Now that you understand Rust's ownership model, you can read real bioinformatics Rust code. This notebook shows three core sequence analysis algorithms — FASTA parsing, k-mer counting, and reverse complement — implemented in Rust. The Python reference implementations run in this notebook; the Rust code is shown as strings for study.

The goal: after this notebook, you can read noodles or needletail source code and understand what every major construct does.

---

## 2. Intuition

**FASTA parsing in Rust:** A FASTA file is streamed line by line via `BufReader`. The parser maintains a minimal state machine: header line (starts with `>`) or sequence line. No whole-file allocation — one sequence at a time.

**K-mer counting in Rust:** A `HashMap<String, usize>` accumulates counts. The `entry().or_insert(0)` pattern is the idiomatic Rust way to increment a counter — no branching on key existence.

**Reverse complement in Rust:** Iterate over bytes in reverse, apply a match statement to complement each base. The result is a new `Vec<u8>` — no in-place mutation required.

---

## 3. Biological background

**FASTA format:**
```
>seq1 Homo sapiens chromosome 22
ATCGATCGATCG...
>seq2
GCTAGCTAGCTA...
```

The `>` line is the header (ID + description). Every subsequent line until the next `>` is sequence. Sequences can span multiple lines.

**K-mers** are all subsequences of length k. For k=3, 'ATCG' has 2 3-mers: 'ATC' and 'TCG'. K-mer frequency spectra are used for genome assembly quality assessment, taxonomic classification (kraken2 uses 31-mers), and repeat detection.

**Reverse complement:** DNA is double-stranded. The complement of each base (A↔T, C↔G) read in reverse gives the sequence on the opposite strand. In genome analysis, k-mers are typically counted on both strands — so each k-mer and its reverse complement are treated as the same feature.

---

## 4. Mathematical explanation

### K-mer spectrum

For a sequence of length $L$ and k-mer size $k$:
- Number of k-mers: $L - k + 1$
- Maximum possible distinct k-mers: $4^k$ (for DNA)
- At k=21: $4^{21} \approx 4 \times 10^{12}$ — more distinct k-mers than bases in the human genome, so 21-mers are nearly unique genome-wide

### Reverse complement

For sequence $s = s_1 s_2 \ldots s_L$, the reverse complement is:

$$\text{RC}(s) = \overline{s_L} \overline{s_{L-1}} \ldots \overline{s_1}$$

where $\overline{A} = T$, $\overline{T} = A$, $\overline{C} = G$, $\overline{G} = C$.

Canonical k-mer: $\min(\text{kmer}, \text{RC}(\text{kmer}))$ lexicographically — this is how strand-agnostic counting works.

---

## 5. Computational explanation — Rust code for all three algorithms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import time
import io

np.random.seed(42)
print("NB09: Rust Sequences, K-mers, and FASTA Parsing")

In [ ]:
# --- Rust: FASTA parser ---

rust_fasta_parser = '''
use std::io::{BufRead, BufReader};
use std::fs::File;

#[derive(Debug)]
struct FastaRecord {
    id: String,
    sequence: String,
}

fn parse_fasta(path: &str) -> Vec<FastaRecord> {
    let file = File::open(path).expect("Cannot open FASTA file");
    let reader = BufReader::new(file);  // buffered reading — reads 8KB at a time
    
    let mut records: Vec<FastaRecord> = Vec::new();
    let mut current_id: Option<String> = None;
    let mut current_seq: String = String::new();
    
    for line in reader.lines() {
        let line = line.expect("IO error reading line");  // unwrap Result<String, Error>
        let line = line.trim();                           // remove trailing whitespace
        
        if line.starts_with(\'>\') {
            // New header: save previous record if any
            if let Some(id) = current_id.take() {  // take() = move out of Option
                records.push(FastaRecord {
                    id,
                    sequence: current_seq.clone(),
                });
                current_seq.clear();
            }
            // Parse the new ID (everything after \'>\')
            current_id = Some(line[1..].split_whitespace()
                .next()
                .unwrap_or("")
                .to_string());
        } else if !line.is_empty() {
            // Sequence line: append to current sequence
            current_seq.push_str(line);
        }
    }
    
    // Don\'t forget the last record
    if let Some(id) = current_id {
        records.push(FastaRecord { id, sequence: current_seq });
    }
    
    records
}
'''

print("Rust FASTA parser:")
print(rust_fasta_parser)
print("Key Rust concepts used:")
print("  BufReader       — buffered IO, reads chunks rather than byte-by-byte")
print("  Option<String>  — Some(id) or None, no null pointers")
print("  .take()         — moves out of an Option, leaving None in its place")
print("  if let Some(x)  — pattern match to unwrap an Option")
print("  .expect()       — panics with a message if Result is Err")

In [ ]:
# --- Python reference implementation: FASTA parser ---

def parse_fasta_python(fasta_text: str) -> list[dict]:
    """Parse a FASTA string into a list of {id, sequence} dicts."""
    records = []
    current_id = None
    current_seq = []
    
    for line in fasta_text.splitlines():
        line = line.strip()
        if line.startswith('>'):
            if current_id is not None:
                records.append({'id': current_id, 'sequence': ''.join(current_seq)})
                current_seq = []
            current_id = line[1:].split()[0]
        elif line:
            current_seq.append(line)
    
    if current_id is not None:
        records.append({'id': current_id, 'sequence': ''.join(current_seq)})
    
    return records

# Test with a synthetic FASTA
synthetic_fasta = """>seq1 Homo sapiens test sequence
ATCGATCG
CGTAGCTA
>seq2
GCTAGCTAGCTA
>seq3 E. coli K-12
AAAATTTTCCCCGGGG"""

records = parse_fasta_python(synthetic_fasta)
for r in records:
    print(f"  {r['id']}: {r['sequence'][:20]}... (len={len(r['sequence'])})")

In [ ]:
# --- Rust: k-mer counter using HashMap ---

rust_kmer_counter = '''
use std::collections::HashMap;

fn count_kmers(sequence: &str, k: usize) -> HashMap<String, usize> {
    let mut counts: HashMap<String, usize> = HashMap::new();
    let bytes = sequence.as_bytes();
    
    for i in 0..bytes.len().saturating_sub(k - 1) {
        // bytes[i..i+k] is a byte slice — no allocation
        // to_string() creates an owned String for the HashMap key
        let kmer = std::str::from_utf8(&bytes[i..i+k])
            .unwrap_or("")
            .to_string();
        
        // entry().or_insert(0): get existing value or insert 0, then increment
        *counts.entry(kmer).or_insert(0) += 1;
    }
    
    counts
}

// Canonical k-mer (strand-agnostic counting)
fn canonical_kmer(kmer: &str) -> String {
    let rc = reverse_complement(kmer);
    // Return the lexicographically smaller of kmer and its RC
    if kmer <= rc.as_str() {
        kmer.to_string()
    } else {
        rc
    }
}
'''

print("Rust k-mer counter:")
print(rust_kmer_counter)

# Python reference implementation
def count_kmers_python(sequence: str, k: int) -> dict[str, int]:
    """Count all k-mers in a sequence."""
    counts: dict[str, int] = defaultdict(int)
    for i in range(len(sequence) - k + 1):
        counts[sequence[i:i+k]] += 1
    return dict(counts)

def canonical_kmer(kmer: str) -> str:
    """Return the lexicographically smaller of kmer and its reverse complement."""
    rc = reverse_complement(kmer)
    return min(kmer, rc)

# Test
seq = "ATCGATCGATCG"
kmers_3 = count_kmers_python(seq, k=3)
print(f"\n3-mers of '{seq}':")
for kmer, count in sorted(kmers_3.items()):
    print(f"  {kmer}: {count}")

In [ ]:
# --- Rust: reverse complement using byte arrays ---

rust_revcomp = '''
fn reverse_complement(seq: &str) -> String {
    seq.bytes()                           // iterate over bytes
        .rev()                            // reverse the iterator
        .map(|b| complement_base(b))      // complement each base
        .collect::<Vec<u8>>()             // collect into Vec<u8>
        .iter()
        .map(|&b| b as char)
        .collect()                        // collect into String
}

fn complement_base(b: u8) -> u8 {
    match b {
        b\'A\' => b\'T\',
        b\'T\' => b\'A\',
        b\'C\' => b\'G\',
        b\'G\' => b\'C\',
        b\'a\' => b\'t\',
        b\'t\' => b\'a\',
        b\'c\' => b\'g\',
        b\'g\' => b\'c\',
        _     => b,         // N and other ambiguous bases unchanged
    }
}
'''

print("Rust reverse complement:")
print(rust_revcomp)

# Python reference implementation
COMPLEMENT = str.maketrans('ATCGatcgNn', 'TAGCtagcNn')

def reverse_complement(seq: str) -> str:
    """Reverse complement of a DNA sequence."""
    return seq.translate(COMPLEMENT)[::-1]

# Test cases
test_seqs = ['ATCG', 'AAAA', 'GCTAGC', 'ATCGN']
print("\nPython reverse complement:")
for s in test_seqs:
    rc = reverse_complement(s)
    print(f"  {s} → {rc}")

# Verify: RC of RC = original
for s in test_seqs:
    assert reverse_complement(reverse_complement(s)) == s, f"RC(RC({s})) != {s}"
print("  RC(RC(s)) == s for all test sequences: PASSED")

In [ ]:
# --- Benchmark Python implementations ---

rng = np.random.default_rng(42)

def gen_dna_seq(length, rng):
    return ''.join(rng.choice(list('ATCG'), size=length))

seq_100 = gen_dna_seq(100, rng)
seq_1000 = gen_dna_seq(1000, rng)
seq_10000 = gen_dna_seq(10000, rng)

import timeit

# K-mer counting
t_kmer = {
    100:   timeit.timeit(lambda: count_kmers_python(seq_100, 21), number=1000) / 1000,
    1000:  timeit.timeit(lambda: count_kmers_python(seq_1000, 21), number=100) / 100,
    10000: timeit.timeit(lambda: count_kmers_python(seq_10000, 21), number=20) / 20,
}

# Reverse complement
t_rc = {
    100:   timeit.timeit(lambda: reverse_complement(seq_100), number=10000) / 10000,
    1000:  timeit.timeit(lambda: reverse_complement(seq_1000), number=1000) / 1000,
    10000: timeit.timeit(lambda: reverse_complement(seq_10000), number=100) / 100,
}

print(f"{'Length':>8} {'K-mer(21) time':>18} {'RevComp time':>15}")
for L in [100, 1000, 10000]:
    print(f"{L:>8,} {t_kmer[L]*1e6:>15.1f} μs {t_rc[L]*1e6:>15.1f} μs")

print()
print("Expected Rust speedup (from needletail benchmarks):")
print("  FASTQ parsing: ~10x over Python")
print("  K-mer counting: ~5-20x over Python dict defaultdict")
print("  Reverse complement: ~3-5x over Python str.translate")

## 7. Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: K-mer frequency spectrum
ax = axes[0]
long_seq = gen_dna_seq(10000, rng)
kmers_21 = count_kmers_python(long_seq, 21)
counts = list(kmers_21.values())
ax.hist(counts, bins=range(1, max(counts)+2), color='#2980b9', edgecolor='black', alpha=0.8)
ax.set_xlabel('K-mer count', fontsize=11)
ax.set_ylabel('Number of distinct 21-mers', fontsize=11)
ax.set_title('21-mer Frequency Spectrum\n(random 10kb sequence)', fontweight='bold')
ax.text(0.6, 0.8, f'Total distinct: {len(kmers_21)}',
        transform=ax.transAxes, fontsize=10)
ax.grid(True, axis='y', alpha=0.3)

# Panel 2: Benchmark comparison table
ax = axes[1]
ax.axis('off')
lengths = [100, 1000, 10000]
rust_speedup_kmer = [5, 10, 15]  # estimated from needletail literature
table_data = []
col_labels = ['Length', 'Python (μs)', 'Rust est. (μs)', 'Speedup']
for L, rs in zip(lengths, rust_speedup_kmer):
    py_us = t_kmer[L] * 1e6
    rust_us = py_us / rs
    table_data.append([f'{L:,}', f'{py_us:.1f}', f'{rust_us:.1f}', f'{rs}x'])
table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center'
)
table.auto_set_font_size(True)
table.scale(1, 2)
ax.set_title('K-mer Counting (k=21):\nPython vs Estimated Rust', fontweight='bold')

# Panel 3: Rust vs Python algorithm structure
ax = axes[2]
ax.axis('off')
struct_text = (
    "Algorithm Structure: K-mer Counting\n\n"
    "Python:\n"
    "  seq = 'ATCG...'  (Python str)\n"
    "  counts = defaultdict(int)\n"
    "  for i in range(L - k + 1):\n"
    "    kmer = seq[i:i+k]  (new str object)\n"
    "    counts[kmer] += 1\n"
    "  overhead: str slice allocation per iter\n\n"
    "Rust:\n"
    "  let bytes = seq.as_bytes();  (no copy)\n"
    "  let mut counts = HashMap::new();\n"
    "  for i in 0..L-k+1 {\n"
    "    // &bytes[i..i+k] is a SLICE\n"
    "    // no allocation unless HashMap needs it\n"
    "    *counts.entry(...).or_insert(0) += 1;\n"
    "  }\n"
    "  advantage: fewer allocations,\n"
    "  faster hash, C-speed loop"
)
ax.text(0.02, 0.98, struct_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='#eaf2f8', alpha=0.8))
ax.set_title('Python vs Rust:\nK-mer Algorithm Structure', fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/nb09_rust_sequences.png', dpi=100, bbox_inches='tight')
plt.show()

## 8. Exercises

1. **Python FASTA parser with multi-line sequences:** Modify `parse_fasta_python` to handle sequences split across many lines (the standard case). Verify on a 3-record FASTA where each sequence spans 5 lines.

2. **Canonical k-mers:** Implement `count_kmers_canonical(seq, k)` that counts each k-mer only in its canonical form (min of kmer, RC(kmer)). Compare the distinct k-mer count to non-canonical counting for a 1000-base random sequence.

3. **Benchmark dict vs defaultdict:** Compare `dict` with `setdefault`, `defaultdict(int)`, and `Counter` for k-mer counting at N=10,000 and k=21. Which is fastest? Why?

4. **Rust reading:** Read the needletail `src/parser.rs` on GitHub (https://github.com/onecodex/needletail). Identify: (a) the struct that represents a parsed record, (b) where the lifetime annotation appears, (c) how the FASTA vs FASTQ format is distinguished.

## 12. Reflection

*Write here: What is the key algorithmic difference between the Python and Rust k-mer counter in terms of memory allocation per iteration? Why does needletail use `&str` slices rather than `String` for record fields, and what would go wrong if it used `String`?*

---

## Papers referenced

- Irber, L. et al. (2022). "Lightweight compositional analysis of metagenomes with FracMinHash." *eLife* 11:e79849.

## References

- needletail: https://github.com/onecodex/needletail
- noodles FASTA reader: https://docs.rs/noodles-fasta/
- rust-bio k-mer module: https://docs.rs/bio/latest/bio/

## Future work / open questions

- How would you extend this Rust FASTA parser to handle FASTQ (which has quality scores)? What new struct fields would you need?
- Minimizers are a more efficient alternative to raw k-mers for metagenomic classification. How is a minimizer defined, and how would you implement it in Python? In Rust?